# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** select your **`aerolex-rag`**, **`.venv`**, or existing environment's kernel, then run top to bottom.
Everything works offline (extractive answers); with a `GEMINI_API_KEY` you get full
generation. You built each stage? The final section assembles them with the
`aerolex` library.


## 0 · Setup

In [9]:
import os
import time
import numpy as np
from pathlib import Path
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.embeddings import GeminiEmbedder
from aerolex.chunking import chunk_corpus, load_chunks
from aerolex.store import InMemoryVectorStore

In [2]:
# Set API key, LLM, Embedding Model 
EMBED_MODEL = "gemini-embedding-001"
EMBED_DIM = 768
GEN_MODEL = "gemini-3.5-flash-lite"
KEY = config.load_api_key()
CACHE_DIR = r"C:\src\IEEEWorkshop\aerolex-production-rag\cache"

## 1 · Hierarchical Chunking 

Legal text is a **hierarchy** (Article / § / RULE → paragraph). We split on those
*logical* boundaries — never fixed-size windows — and attach citation metadata, so
an answer can cite *"EU 261/2004 — Article 7"* instead of *"chunk 42"*.

In [3]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

# Prefer the pre-built cache (instant); otherwise chunk live.
try:
    chunks = load_chunks()
    print(f"\nloaded {len(chunks)} chunks from cache")
except FileNotFoundError:
    chunks = chunk_corpus(docs)
    print(f"\nchunked live: {len(chunks)} chunks")

by_id = {c.chunk_id: c for c in chunks}
art7 = next(c for c in chunks if c.section_label == "Article 7")
print("\nExample chunk —", art7.citation)
print(art7.text[:320], "…")

delta_domestic   delta   78,663 chars  Delta Domestic General Rules Tariff (Contract of Carriage)
delta_intl       delta  118,099 chars  Delta International General Rules Tariff (Contract of Carriage)
eu261            eu      27,544 chars  EU Regulation (EC) No 261/2004
cfr260           cfr     26,287 chars  US DOT 14 CFR Part 260

loaded 234 chunks from cache

Example chunk — EU Regulation (EC) No 261/2004 — Article 7
1.
Where reference is made to this Article, passengers shall
receive compensation amounting to:
(a) EUR 250 for all flights of 1 500 kilometres or less;
(b) EUR 400 for all intra-Community flights of more than
1 500 kilometres, and for all other flights between 1 500
and 3 500 kilometres;
(c) EUR 600 for all flights no …


# 3 · Embedding Creation

In [7]:
if not KEY:
    print("GEMINI_API_KEY not set.")

print(f"Embedding {len(chunks)} chunks with {EMBED_MODEL} "
        f"(dim={EMBED_DIM}) …")

t0 = time.time()
embedder = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM)
matrix = embedder.embed_documents([c.searchable_text for c in chunks])
print(f"Embedded in {time.time() - t0:.1f}s, shape={matrix.shape}")

Embedding 234 chunks with gemini-embedding-001 (dim=768) …
Embedded in 135.2s, shape=(234, 768)


In [10]:
store = InMemoryVectorStore([c.chunk_id for c in chunks], matrix)
store.save(Path(CACHE_DIR) / "new_embeddings.npz")

In [ ]:
# if time out errors load, already created vector store
# store = InMemoryVectorStore.load(Path(CACHE_DIR) / "new_embeddings.npz")
# print (f"{store.matrix.shape[0]} Embeddings loaded successfully of shape {store.matrix.shape[1]}!")

## 2 · Lexical Search with BM25  ·  Feature #2

BM25 (Best Match 25) is a probabilistic ranking algorithm to measure how relevant a document is to a query, combining term frequency, inverse document frequency, and document‑length normalization.
Use `BM25Okapi` directly with the same stopword-filtering tokenizer for chunks and queries. Scores are lexical relevance scores, not cosine similarities.


### 🛠️ TODO: write `tokenize`

In [ ]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Keep words and dotted/slashed numeric refs as single tokens.
_TOKEN = re.compile(r"[a-z]+|\d+(?:[./-]\d+)*")


def tokenize(text: str) -> list[str]:
    # TODO Fetch list of keywords
    # TODO Remove the stop words
    return None

assert tokenize("What is the refund for a cancelled flight?") == ["refund", "cancelled", "flight"]
print("tokenize OK")

In [19]:
# Tokenize the query for BM25 search
query_tokens = tokenize("What is the refund for a cancelled flight")

In [20]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search(query_tokens, top_k=5):
    print(f"{score:6.2f}  {by_id[cid].citation}")

  5.21  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
  4.72  US DOT 14 CFR Part 260 — § 260.1
  4.67  US DOT 14 CFR Part 260 — § 260.6 (part 6/6)
  4.66  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
  4.47  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)


## 3 · Dense Semantic Search + In-Memory Store  ·  Features #3 & #4

Gemini embeds every chunk into a vector; the whole corpus is one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.


### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [ ]:
def cosine_search(query_vec, matrix, ids, top_k=5):

    # TODO: 1) L2-normalize query_vec and each row of matrix

    #       2) scores = matrix @ query   (cosine, since normalized)
    #       3) return the top_k (id, score) pairs, highest first
    return None

# sanity check: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "What is the refund for a cancelled flight"

query_vector = GeminiEmbedder(api_key=KEY, model=EMBED_MODEL, dim=EMBED_DIM).embed_query(query)

dense_hits = cosine_search(query_vector, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

0.726  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.724  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.713  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.698  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)
0.696  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)


## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine scores live on different scales, so we fuse by **rank**:
each list contributes `1 / (k + rank + 1)` to an id's score (`k = 60`). Documents that
rank well in *either* retriever bubble up.

### 🛠️ TODO: write `rrf`

In [ ]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            chunk_score = scores.get(cid, 0.0)
            # TODO for each chunk compute the RRF Score
            # scores[cid] = ?
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [24]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

Fused top-5:
0.0323  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
0.0303  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
0.0296  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
0.0161  US DOT 14 CFR Part 260 — § 260.1
0.0159  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)


## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [25]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
final_chunks = [by_id[cid] for cid, _ in reranked]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Re-ranked top chunks:
   6.68  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 1/6)
   6.28  US DOT 14 CFR Part 260 — § 260.6 (part 4/6)
   5.77  US DOT 14 CFR Part 260 — § 260.6 (part 1/6)
   5.55  US DOT 14 CFR Part 260 — § 260.6 (part 5/6)
   5.28  US DOT 14 CFR Part 260 — § 260.6 (part 2/6)
   5.24  Delta International General Rules Tariff (Contract of Carriage) — RULE 23 (part 1/7)
   3.78  US DOT 14 CFR Part 260 — § 260.1
   1.71  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 5/6)
   0.84  Delta International General Rules Tariff (Contract of Carriage) — RULE 15 (part 4/4)
  -0.09  Delta Domestic General Rules Tariff (Contract of Carriage) — RULE 22 (part 2/6)


## 6 · Grounded Answer with Inline Citations  ·  Feature #7

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [26]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    used = []
    def keep(m):
        n = int(m.group(1))
        if n in index:
            if n not in used:
                used.append(n)
            return m.group(0)
        return ""                      # drop a citation to a non-existent passage
    cleaned = CITE.sub(keep, answer)
    return cleaned, [f"[[{n}]] {index[n]}" for n in sorted(used)]

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
print(_c, "|", _r)

A [[1]] and B . | ['[[1]] EU261 Art 7']


In [ ]:
from aerolex.generate import Generator

result = Generator(model = GEN_MODEL).generate(query, final_chunks)
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

## 7 · Conversational Memory  ·  Feature #8

Follow-ups like *"what about in the US?"* are meaningless to a retriever alone. We
**condense** the chat history + follow-up into a standalone query before searching.

In [16]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result["answer"][:160])
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

follow-up : what about in the US?
standalone: How much compensation for a long delayed flight in the US?


## 8 · Long-Term Persistence  ·  Feature #9

Facts a user states persist across sessions in SQLite and can be injected into
future prompts.

In [17]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

Known facts about the user:
- home_airport: JFK
- status: Delta Platinum Medallion


## 10 · The Full Pipeline

Now assemble every stage with the `aerolex` library — the productionized version of
what you just built — and ask a question end to end.

In [19]:
from aerolex.pipeline import RagPipeline

if key:
    from aerolex.embeddings import GeminiEmbedder
    pipe = RagPipeline(
        chunks=chunks,
        store=store,
        bm25=bm25,
        embedder=GeminiEmbedder(),
        reranker=CrossEncoderReranker(),
        generator=Generator(),
    )
    out = pipe.answer("If my 4000 km flight is cancelled, what am I owed?",
                      memory=mem, profile=prof, user_id="demo_user")
    print(out["answer"])
    print("\nReferences:")
    for r in out["references"]:
        print("  ", r)
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9937.31it/s]


What you are owed depends on the applicable regulations and airline terms governing your flight:

**Under EU Regulation (EC) No 261/2004 (if applicable):**
* Because a 4,000 km flight exceeds 3,500 km, it falls under Article 7(1)(c), which provides compensation of EUR 600 [[1]].
* The operating air carrier may reduce this compensation by 50% if you are offered re-routing to your final destination on an alternative flight that arrives no more than four hours after your original scheduled arrival time [[1]].
* Compensation is payable in cash, by electronic bank transfer, bank orders, bank cheques, or, with your signed agreement, in travel vouchers and/or other services [[1]].
* In addition to compensation, you must be offered assistance in accordance with Article 8 and Article 9 [[5]].
* Compensation is not required if you were informed of the cancellation at least two weeks prior to departure, or if you were given notice between two weeks and seven days prior with re-routing departing n